# Beginner Machine Learning Practice — SCM Stock Status

**Audience:** Beginner with little or no machine-learning knowledge  
**Environment:** Databricks notebook using Python, pandas, and scikit-learn  
**Time:** Work at your own pace; run one cell at a time  
**Purpose:** Understand the basic machine-learning workflow through a small, understandable inventory example.

## Business problem

Imagine a computer manufacturer wants to classify an inventory item as `Low` or `Normal` based on its available quantity. We will create a tiny synthetic dataset, teach a model from labelled examples, ask it to predict labels for new quantities, and measure its performance.

This is a learning exercise, not a real inventory policy. The sample labels are invented for practice, and quantity alone is not enough for a production stock-risk decision. A real system may need demand, supplier lead time, open orders, safety stock, seasonality, and other information.

## Learning outcomes

By the end, you should be able to:
1. Explain what machine learning does in this example.
2. Distinguish a dataset, a feature (`X`), and a target (`y`).
3. Explain why data is explored before training.
4. Explain training/test split, `fit()`, and `predict()`.
5. Read accuracy and a confusion matrix at a basic level.
6. Change the data and observe how the model's results can change.
7. Identify what this notebook does **not** cover yet (MLflow, model registry, deployment).

## Roadmap

1. Create labelled examples
2. Explore the data (EDA)
3. Separate feature and target
4. Split data into training and test sets
5. Train a decision-tree classifier
6. Predict new examples
7. Evaluate predictions
8. Practice changing data and model settings
9. Review key terms and connect this exercise to the course

**Please do not rush.** Your goal is to understand each step, not to finish a course or memorize every line.

## Before coding: what is machine learning?

In traditional programming, a developer writes explicit rules. For example, a rule might say: “if quantity is below 15, label it Low.” In machine learning, we give an algorithm examples with known answers, and it learns a pattern from those examples.

In this notebook:
- **Examples/rows:** individual item observations.
- **Feature/input:** `available_quantity`.
- **Label/target:** `stock_status`, the known answer.
- **Model:** the trained object that learns a relationship between input and label.
- **Prediction:** the model's answer for an input it has not been given a label for in that prediction step.
- **Evaluation:** comparing predictions against known correct answers on held-out data.

### Classification vs. regression
- **Classification** predicts a category, such as `Low` or `Normal`.
- **Regression** predicts a numeric value, such as expected demand next week.

We use classification here because `stock_status` is a category.

## Cell 1 — Create sample data

Each row is one example. The `stock_status` column is the answer we provide during training.

In [0]:
import pandas as pd

data = {
    "available_quantity": [3, 5, 8, 10, 12, 15, 20, 25, 30, 40, 50, 60],
    "stock_status": [
        "Low", "Low", "Low", "Low", "Low", "Low",
        "Normal", "Normal", "Normal", "Normal", "Normal", "Normal"
    ]
}

df = pd.DataFrame(data)
display(df)

### What the code means
"""
- `import pandas as pd` loads the pandas library and gives it the short name `pd`.
- `data` is a Python dictionary: each key becomes a column name, and each list supplies that column's values.
- `pd.DataFrame(data)` turns the dictionary into a table.
- `display(df)` shows the table in Databricks.
"""

### Expected outcome
"""
You should see 12 rows and 2 columns. Six rows are labelled `Low`, and six are labelled `Normal`. The rows are synthetic examples created for this lesson.
"""

**Understand the outcome:** `df` is our labelled example data. Real ML training needs examples where the correct answer is known. Here, we created the answers ourselves so we can practice safely.

## Cell 2 — Explore the data (EDA)

EDA means looking at the data before training. Check row counts, data types, label distribution, and numeric summaries.

### How to interpret the results
- `df.shape` returns `(rows, columns)`.
- `df.info()` shows column names, data types, and non-null counts.
- `value_counts()` counts how many rows belong to each stock-status category.
- `describe()` gives numeric summaries such as count, mean, standard deviation, minimum, quartiles, and maximum.

### Why EDA matters
EDA can reveal missing values, unexpected data types, extreme values, duplicated rows, and imbalanced labels before training. A model cannot automatically know whether the input data is correct.

In [0]:
print("Rows and columns:", df.shape)
print("\nData types and non-null values:")
df.info()

print("\nExamples per status:")
print(df["stock_status"].value_counts())

print("\nQuantity summary:")
display(df["available_quantity"].describe().to_frame())

**Questions to answer:**
- How many rows are in this dataset?
- How many are labelled `Low`?
- What is the minimum and maximum quantity?
- Why might a model struggle if the examples were missing or incorrectly labelled?

### Why the syntax is different
- `df[["available_quantity"]]` uses double brackets and returns a two-dimensional DataFrame, which is the expected shape for model features.
- `df["stock_status"]` returns a one-dimensional Series for the target.
- `X` is often used as the conventional name for features; `y` is commonly used for the target.

The model learns a relationship between `X` and `y`; it should not receive the target as an input feature when predicting.

## Cell 3 — Separate features and target

`X` is the information the model is allowed to use. `y` is the answer we want it to learn to predict.

In [0]:
X = df[["available_quantity"]]
y = df["stock_status"]

print("Features (X):")
display(X.head())

print("Target (y):")
display(y.head())

### Why not train and test on the same rows?
"""
If we evaluate a model using only the examples it learned from, the score may look too optimistic. A test set gives us a basic check on examples held aside from training.

- `X_train`, `y_train`: examples used to learn.
- `X_test`, `y_test`: examples held aside for evaluation.
- `stratify=y`: tries to preserve the label proportions in both sets.
- `random_state=42`: makes the split reproducible in repeated runs with the same data and environment.

With only 12 rows, the test set is extremely small, so the score is only a teaching illustration.
"""

**Key idea:** We keep the feature as a DataFrame (`df[[...]]`) because scikit-learn expects a two-dimensional input matrix. The target is a Series (`df[...]`).

## Cell 4 — Split training and test examples

Training examples are used to fit the model. Test examples are held aside to estimate how it performs on examples it did not train on.

### What happens during `fit()`?
`DecisionTreeClassifier` is an algorithm for predicting categories. A decision tree learns split rules from the training examples. `max_depth=2` limits the tree's depth so the model stays relatively simple for this first lesson.

- `DecisionTreeClassifier(...)` creates the model object and configures it.
- `model.fit(X_train, y_train)` learns from the training inputs and known labels.
- `random_state=42` makes any randomness reproducible.

The model is not a human and does not understand inventory in a business sense. It learns statistical patterns from the examples supplied.

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("\nTest labels:")
print(y_test.value_counts())

`test_size=0.25` reserves about one quarter of the examples for testing. `random_state=42` makes the split repeatable. `stratify=y` tries to preserve the proportions of the labels in each split.

This dataset is tiny, so the evaluation score can change noticeably with different splits.

### What happens during `predict()`?
`model.predict(new_items)` applies the learned pattern to each row in `new_items` and returns a predicted category. We add those predictions to a copy of the input DataFrame so the input and output can be read side by side.

The model is not being told the correct status for these new quantities in this cell. It estimates a label based on its training.

## Cell 5 — Train a simple model

A decision tree learns rules by splitting the data. We limit its depth to keep this first example simple.

In [0]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=2, random_state=42)
model.fit(X_train, y_train)

print("Training complete.")

### Metrics in simple language
"""
- **Accuracy:** number of correct test predictions divided by total test predictions.
- **Confusion matrix:** a table comparing actual labels with predicted labels. In the code, the order is explicitly `Low`, then `Normal`.
- **Precision:** when the model predicts a class, how often that prediction is correct.
- **Recall:** of all actual examples in a class, how many the model identifies.
- **F1-score:** a combined measure of precision and recall.

For stock-risk use cases, a false `Normal` prediction for an item that is actually at risk could be more costly than a false `Low` prediction. That is why accuracy alone may not be enough in a real system.

The dataset is deliberately small and synthetic. Do not use its score to judge a production model.
"""

**What `fit()` does:** it uses the training features and their known labels to learn a pattern. We are not manually writing an `if quantity < ...` rule; the algorithm chooses splits from the examples.

## Cell 6 — Predict for new quantities

Now we give the trained model quantities for which we have not supplied a status in this cell.

In [0]:
new_items = pd.DataFrame({"available_quantity": [6, 18, 35]})
predictions = model.predict(new_items)

results = new_items.copy()
results["predicted_stock_status"] = predictions
display(results)

**Outcome:** each new quantity gets a predicted category. The model is making a learned prediction, not looking up a label from `df`.

Try changing the quantities to `[2, 22, 45]` and run this cell again.

## Cell 7 — Evaluate on held-out test data

Accuracy is the fraction of test predictions that are correct. A confusion matrix shows correct and incorrect predictions by class.

In [0]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

test_predictions = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, test_predictions), 3))
print("\nConfusion matrix (class order: Low, Normal):")
print(confusion_matrix(y_test, test_predictions, labels=["Low", "Normal"]))

print("\nClassification report:")
print(classification_report(y_test, test_predictions, zero_division=0))

**How to interpret this:**
- Accuracy ranges from 0 to 1; for example, `0.75` means 75% of this test split was predicted correctly.
- The confusion matrix helps reveal which class the model gets wrong.
- Do not treat a high score on this tiny, artificial dataset as evidence that it will work well in production.

## Cell 8 — Practice changing the examples

Change one or more quantities or labels in Cell 1, then rerun the notebook from Cell 1 downward. Observe how the model and its evaluation change.

Suggested experiments:
1. Change the quantity `15` to `35` but leave its label as `Low`. What does this tell you about data quality?
2. Add examples from a wider range of quantities.
3. Remove some examples and compare results.
4. Change `max_depth=2` to `max_depth=1` and compare predictions. A shallower tree is simpler, but may miss patterns.

**Important:** If you change the dataset, rerun all cells in order so `X`, `y`, the split, and the model all use the updated data.

## Final recap

After this practice, you should be able to explain:

- **Dataset:** examples used for learning.
- **Feature (`X`):** input information.
- **Target (`y`):** answer to predict.
- **Training (`fit`):** learning patterns from labelled examples.
- **Prediction (`predict`):** applying the learned model to new inputs.
- **Evaluation:** checking predictions against known test answers.

### How this connects to the Academy material

This notebook focuses on the foundations needed before the course's later steps:
- EDA and feature preparation
- Model training and evaluation
- MLflow tracking and model management (later)
- Mosaic AI Model Serving and workflows (later)

Stop here for now. Your goal is understanding the flow, not completing every topic in one sitting.

## Detailed reference: what each stage achieves

| Stage | What we do | Why we do it | Output |
|---|---|---|---|
| Create data | Build a small table with quantities and known labels | A supervised model needs examples with answers | `df` |
| EDA | Inspect shape, types, labels, and summary statistics | Catch issues and understand the data | A basic understanding of the dataset |
| Feature/target split | Put input columns in `X` and answer column in `y` | Tell the algorithm what it may use and what it should learn to predict | `X`, `y` |
| Train/test split | Reserve some rows for testing | Estimate performance on held-out examples | `X_train`, `X_test`, `y_train`, `y_test` |
| Train | Call `model.fit(...)` | Learn a pattern from examples | A fitted model |
| Predict | Call `model.predict(...)` | Produce predictions for new inputs | Predicted categories |
| Evaluate | Compare predictions with known test labels | Understand errors and performance | Accuracy and classification report |

## Expected outcome of this notebook

When you finish, you will have:
- A small labelled dataset.
- A trained decision-tree classification model.
- Predictions for three new stock quantities.
- Basic evaluation metrics on a held-out test set.
- Hands-on experience changing data and model settings.

You will **not** have:
- A validated inventory forecasting system.
- A model trained on your one-million-item SCM table.
- MLflow experiment tracking.
- A registered model or serving endpoint.
- A production-ready recommendation for inventory replenishment.

Those are later learning steps.

## Common beginner questions

### Does the model know the stock status automatically?
No. During training, we provide examples with known labels. The model learns a pattern from those examples.

### Is `Low` a universal stock threshold?
No. The labels in this notebook are made up for practice. Real thresholds depend on demand, lead time, safety stock, service-level targets, and business rules.

### Why do we need a test set?
To check predictions on examples that were not used to fit the model. It provides a more useful first estimate than testing only on training data.

### Why can the accuracy change?
The dataset is tiny. Changing the split, examples, or tree depth can change the learned model and its score. On real tasks, use enough representative data and an evaluation design appropriate to the problem.

### Why use a decision tree?
It is a simple model for learning category labels and is often easier to explain to beginners than more complex models.

### Does a high accuracy mean the model is good?
Not necessarily. It depends on the dataset, label quality, class balance, error costs, and whether test data represents future cases. In real work, compare multiple relevant metrics and validate on representative data.

### Can I use this directly on my million-row Databricks dataset?
Not yet. This notebook intentionally uses a tiny pandas DataFrame so the workflow is easy to understand. Scaling to a real Spark dataset requires additional data preparation, suitable evaluation, and model workflow choices.

## Practice tasks with expected learning

1. **Change one quantity but not its label.**  
   Learning: inconsistent labels can confuse the model. In real projects, data quality is critical.

2. **Add examples near the boundary between `Low` and `Normal`.**  
   Learning: examples near ambiguous cases influence what the model learns.

3. **Change `max_depth=2` to `max_depth=1`.**  
   Learning: simpler models may be easier to interpret but may miss patterns.

4. **Change the test split.**  
   Learning: small test sets can produce unstable metrics. Keep `random_state` fixed when comparing one change at a time.

5. **Explain the workflow without looking at code.**  
   Learning: understanding the process is more valuable than memorizing syntax.

After changing the original dataset, rerun the notebook from the first code cell in order. Otherwise, later variables may still refer to old data.

## Glossary

- **Dataset:** a collection of examples arranged in rows and columns.
- **Row/example:** one observation.
- **Feature:** an input variable used by a model.
- **Target/label:** the answer to be predicted.
- **Supervised learning:** learning from examples that include known answers.
- **Classification:** predicting a category.
- **Regression:** predicting a numeric value.
- **Training:** fitting a model to examples.
- **Test set:** examples held aside to evaluate the model.
- **Model:** the fitted object that maps inputs to predictions.
- **Prediction:** the model's output for new input data.
- **Metric:** a numeric summary of model performance.
- **Overfitting:** when a model fits training examples too closely and performs poorly on new data.
- **MLflow:** a tool commonly used to track experiments and manage model lifecycle. It is not used in this first notebook.
- **Model serving:** exposing a trained model for inference through a serving interface. It is not used in this first notebook.

## Connection to the Databricks Academy course

This notebook is a gentle foundation for topics such as EDA, feature preparation, model training, and evaluation. The Academy material's later MLflow, model registry, and Mosaic AI Model Serving sections should be approached after these basic concepts make sense.

**Suggested stopping point:** run the notebook, complete at least two practice tasks, and make sure you can explain `X`, `y`, `fit()`, `predict()`, and accuracy in your own words. Do not worry about completing the entire course today.

## Practice

In [0]:
### Requirement: To predict the stock status of a new item

import pandas as pd

#1. Golden Data
data = { "available_quantity":[3,5,8,10,12,15,20,25,30,40,50,60], 
        "stock_status":["low","low","low","low","low","low","normal","normal","normal","normal","normal","normal"]}

df = pd.DataFrame(data)
display(df)

#2. EDA (Exploratiry Data Analysis)
print("\n# Shape (Total Rows and Columns):\n",df.shape)
print("\n# Data Types:\n",df.dtypes)
print("\n# Missing Values:\n",df.isnull().sum())
print("\n# Unique Values:\n",df.nunique())
print("\n# Descriptive Statistics:\n",df.describe())
print("\n# Value Counts:\n",df["stock_status"].value_counts())
print("\n#")

#3. Seperate Model Features(Input) and Target(Output)
X = df[["available_quantity"]]
y = df["stock_status"]

#4. Split Training and Test examples
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42,stratify=y)

print("\n# Training Examples:\n",X_train.shape)
print("\n# Test Examples:\n",X_test.shape)
print("\n#")

#5. Train a simple Model
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier()
model.fit(X_train, y_train)

#6. Test a Model with existing data
predictions = model.predict(X_test)
results = X_test.copy()
results["predicted_stock_status"] = predictions
display(results)

#7. Test a Model with new items
new_items = pd.DataFrame({"available_quantity": [2, 1, 13, 21,100]})
new_items["predicted_stock_status"] = model.predict(new_items)  
display(new_items)

#8. Evaluate on held-out test data
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

test_predictions = model.predict(X_test)

print("\nAccuracy:\n", round(accuracy_score(y_test, test_predictions), 3))
print("\nConfusion matrix (class order: low, normal):\n")
print(confusion_matrix(y_test , test_predictions, labels=["low", "normal"]))

print("\nClassification report:\n")
print(classification_report(y_test, test_predictions, zero_division=0))


Let's use a simple SCM inventory example to explain your classification report in tabular format.

## 1. Example test data

Suppose your ML model predicts whether computer parts have `low` or `normal` stock.

| Record | Computer part | Actual status | Model prediction | Correct?  |
| ------ | ------------- | ------------- | ---------------- | --------- |
| 1      | Keyboard      | Low           | Low              | Yes       |
| 2      | Mouse         | Normal        | Normal           | Yes       |
| 3      | Monitor       | Normal        | Normal           | Yes       |
| Total  |               | 3 records     |                  | 3 correct |

The model predicted all three records correctly.

## 2. Classification report explained

Your output:

```
              precision  recall  f1-score  support
low              1.00    1.00     1.00       1
normal           1.00    1.00     1.00       2
```

| Metric    | Low         | Normal      | Meaning                                                     |
| --------- | ----------- | ----------- | ----------------------------------------------------------- |
| Precision | 1.00 (100%) | 1.00 (100%) | Every prediction for that class was correct.                |
| Recall    | 1.00 (100%) | 1.00 (100%) | Every actual record of that class was identified correctly. |
| F1-score  | 1.00 (100%) | 1.00 (100%) | Combined score of precision and recall.                     |
| Support   | 1           | 2           | Number of actual records in that class.                     |

### 3. How are these numbers calculated?

| Metric    | Calculation for Low                                            | Calculation for Normal                                                 |
| --------- | -------------------------------------------------------------- | ---------------------------------------------------------------------- |
| Precision | 1 correct Low prediction ÷ 1 Low prediction = 100%             | 2 correct Normal predictions ÷ 2 Normal predictions = 100%             |
| Recall    | 1 correctly identified Low record ÷ 1 actual Low record = 100% | 2 correctly identified Normal records ÷ 2 actual Normal records = 100% |
| Support   | 1 actual Low record                                            | 2 actual Normal records                                                |

## 4. Accuracy, macro average and weighted average

| Output           | Value | Explanation                                                           |
| ---------------- | ----- | --------------------------------------------------------------------- |
| Accuracy         | 1.00  | 3 correct predictions ÷ 3 total records = 100%                        |
| Macro average    | 1.00  | Average performance of Low and Normal, treating both classes equally. |
| Weighted average | 1.00  | Average performance weighted by the number of records in each class.  |

In this example, both averages are 100% because all predictions were correct.

## 5. Remember these four terms

| Term      | Easy way to remember                                     |
| --------- | -------------------------------------------------------- |
| Precision | When the model predicts, is it right?                    |
| Recall    | Did the model find all the actual records of that class? |
| F1-score  | One score combining precision and recall.                |
| Support   | How many actual records belong to that class?            |

Important: Your test dataset has only three records. The 100% result applies to these records and does not guarantee the same performance on real-world inventory data.

## Topic 2: Decision Tree Classification and Prediction Rules

### Learning objectives

1. Understand how a Decision Tree makes predictions.
2. Display the decision rules learned by the model.
3. Predict stock status for new inventory quantities.
4. Interpret the model's predictions.
5. Understand the limitations of a small training and test dataset.

### Business scenario

We want to classify computer-part inventory as `low` or `normal` using `available_quantity`.

Our dataset contains 12 records:
- Quantities from 3 to 15 are labelled `low`.
- Quantities from 20 to 60 are labelled `normal`.

The model learns patterns from these examples. It does not automatically know our company's inventory policy.

### Step 1: Understand how a Decision Tree works

A Decision Tree learns conditions from the training data.

For example, it may ask:

- Is the available quantity less than or equal to a particular threshold?
- If yes, which class should be predicted?
- If no, which class should be predicted?

The threshold is learned during training. We will inspect the actual rules learned by our model rather than assume a threshold.

**Key terms**

| Term | Meaning |
|---|---|
| Feature | Input column: `available_quantity` |
| Target | Output column: `stock_status` |
| Decision rule | A condition used to make a prediction |
| Leaf node | The final node that gives a class prediction |
| Prediction | The class returned by the model |

In [0]:
from sklearn.tree import export_text

rules = export_text(
    model,
    feature_names=["available_quantity"]
)

print(rules)

In [0]:
import pandas as pd

new_inventory = pd.DataFrame({
    "available_quantity": [4, 14, 18, 22, 35, 55]
})

predictions = model.predict(new_inventory)

for qty, status in zip(
    new_inventory["available_quantity"],
    predictions
):
    print(f"Quantity: {qty}, Predicted status: {status}")

In [0]:
comparison = X_test.copy()

comparison["actual_status"] = y_test
comparison["predicted_status"] = test_predictions

comparison

### Step 7: Summary

In this session, I learned:

- How a Decision Tree makes decisions using conditions.
- How to display rules with `export_text()`.
- How to predict stock status for new quantities with `predict()`.
- How to compare actual labels with predicted labels.
- Why small test datasets provide limited evidence of model performance.

### Next topic: Overfitting and underfitting

In the next session, I will learn:

- What overfitting means.
- What underfitting means.
- Why training performance and test performance can differ.
- How `max_depth` controls the complexity of a Decision Tree.
- How to compare a model's performance on training and test data.

**Checkpoint:** Run the rules cell, the new-prediction cell, and the comparison cell. Record your actual outputs before moving to the next topic.

## Topic 3: Overfitting and Underfitting

**Objective:** Understand how Decision Tree complexity affects model performance.

- **Underfitting:** Model is too simple.
- **Overfitting:** Model learns training data too closely.
- **Good fit:** Model performs well on unseen data.

We will compare Decision Trees using different `max_depth` values.

In [0]:
from sklearn.metrics import accuracy_score

print("Training accuracy:",
      accuracy_score(y_train, model.predict(X_train)))

print("Test accuracy:",
      accuracy_score(y_test, model.predict(X_test)))

In [0]:
from sklearn.tree import DecisionTreeClassifier
import pandas as pd

models = {
    "Depth 1": DecisionTreeClassifier(max_depth=1, random_state=42),
    "Depth 2": DecisionTreeClassifier(max_depth=2, random_state=42),
    "No limit": DecisionTreeClassifier(max_depth=None, random_state=42)
}

results = []

for name, tree in models.items():
    tree.fit(X_train, y_train)
    results.append({
        "Model": name,
        "Train accuracy": accuracy_score(y_train, tree.predict(X_train)),
        "Test accuracy": accuracy_score(y_test, tree.predict(X_test))
    })

pd.DataFrame(results).round(3)

### What did I learn?

| Observation | Possible meaning |
|---|---|
| Low training and test accuracy | Underfitting |
| High training, much lower test accuracy | Overfitting |
| Both accuracies high and similar | Potentially good fit |

**Note:** Our dataset contains only 12 records, so these results are for practice, not reliable production evaluation.

**Next topic:** Compare Decision Tree with Logistic Regression.

## Topic 4: Comparing ML Models

**Objective:** Train two different classification models using the same SCM inventory dataset and compare their accuracy.

- **Decision Tree** learns decision rules.
- **Logistic Regression** learns a relationship between features and class probabilities.

We will use the same `X_train`, `X_test`, `y_train`, and `y_test` data.

In [0]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=2, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000)
}

for name, model in models.items():
    model.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))

    print(name)
    print("Training accuracy:", round(train_acc, 3))
    print("Test accuracy:", round(test_acc, 3))
    print()

### Key takeaways

| Model | How it works |
|---|---|
| Decision Tree | Uses learned conditions and branches |
| Logistic Regression | Estimates class probabilities using a mathematical relationship |

Compare their test accuracy, but do not automatically assume the higher score means the model is always better.

**Important:** Our dataset has only 12 records, so the comparison is just for learning.

**Next topic:** Feature engineering — how adding useful inventory information can help a model.

## Topic 5: Feature Engineering

**Objective:** Understand how creating useful input features can help a Machine Learning model.

A feature is an input used to make a prediction.

Our current dataset has only one feature: `available_quantity`.

In a real SCM inventory system, stock status may also depend on daily demand and supplier lead time.

We will create two additional example features for practice.

In [0]:
import pandas as pd

df_features = pd.DataFrame({
    "available_quantity": [3, 5, 8, 10, 12, 15, 20, 25, 30, 40, 50, 60],
    "daily_demand":       [5, 4, 6, 3, 4, 5, 2, 3, 2, 4, 3, 2],
    "lead_time_days":     [7, 5, 6, 4, 5, 3, 4, 3, 2, 3, 2, 2],
    "stock_status": [
        "low", "low", "low", "low", "low", "low",
        "normal", "normal", "normal", "normal", "normal", "normal"
    ]
})

df_features["days_of_stock"] = (
    df_features["available_quantity"] / df_features["daily_demand"]
)

df_features

### What did we create?

| Feature | Meaning |
|---|---|
| `available_quantity` | Current inventory quantity |
| `daily_demand` | Example daily demand |
| `lead_time_days` | Example supplier delivery time |
| `days_of_stock` | Approximate days inventory will last at the given demand |

**Formula:**

Days of stock = Available quantity ÷ Daily demand

These additional values are illustrative practice data, not actual supplier or inventory measurements.

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X = df_features[
    ["available_quantity", "daily_demand",
     "lead_time_days", "days_of_stock"]
]

y = df_features["stock_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model_features = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

model_features.fit(X_train, y_train)

# print the values of feature and target columns
prediction = model_features.predict(X_test)
results = X_test.copy()
results["predicted_stock_status"] = prediction
display(results)

print("Training accuracy:",
      round(accuracy_score(
          y_train, model_features.predict(X_train)
      ), 3))

print("Test accuracy:",
      round(accuracy_score(
          y_test, model_features.predict(X_test)
      ), 3))

### Learning outcome

- Feature engineering creates or transforms useful model inputs.
- `days_of_stock` is a derived feature calculated from quantity and demand.
- A model can use multiple features instead of just one.
- Test accuracy alone, especially on only 12 records, cannot prove that extra features improve real-world performance.

**Next topic:** Cross-validation — evaluating model performance across multiple data splits.

## Topic 6: Cross-Validation

**Objective:** Understand how cross-validation evaluates a Machine Learning model using multiple data splits.

With a normal train/test split, the model is evaluated on one test set. With cross-validation, the training data is split into several folds, and the model is trained and evaluated repeatedly.

This helps us understand how consistently a model performs across different splits.

In [0]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier

X = df_features[
    ["available_quantity", "daily_demand",
     "lead_time_days", "days_of_stock"]
]

y = df_features["stock_status"]

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

model_cv = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

scores = cross_val_score(
    model_cv,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Accuracy for each fold:", scores)
print("Average accuracy:", round(scores.mean(), 3))

## Normal Train/Test Split vs. Cross-Validation

| Step | Normal Train/Test Split | 3-Fold Cross-Validation |
|---|---|---|
| Input data | All 12 records | All 12 records |
| Data splitting | One split | Three different splits |
| Training | Train on one training set | Train three times |
| Testing | Test on one held-out set | Each fold is held out once |
| Result | One test accuracy | Three accuracy scores and their average |


##Topic 7: MLflow — Experiment Tracking

### Objective
Understand how MLflow records Machine Learning experiments.

When we train different models, we may want to compare their settings and accuracy.

| Term | Meaning | Example |
|---|---|---|
| Experiment | Organizes related runs | SCM ML practice |
| Run | One experiment execution | Train a Decision Tree |
| Parameter | A model setting | `max_depth = 2` |
| Metric | A measured result | Test accuracy = 0.90 |
| Artifact | A saved output | Trained model file |

### Business scenario
We will train Decision Trees with different `max_depth` values and record their settings and test accuracy using MLflow.

In [0]:
import mlflow
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

mlflow.set_experiment("/Shared/scm-ml-practice")

for depth in [1, 2]:
    with mlflow.start_run(run_name=f"tree_depth_{depth}"):
        model_mlflow = DecisionTreeClassifier(
            max_depth=depth,
            random_state=42
        )

        model_mlflow.fit(X_train, y_train)

        accuracy = accuracy_score(
            y_test,
            model_mlflow.predict(X_test)
        )

        mlflow.log_param("max_depth", depth)
        mlflow.log_metric("test_accuracy", accuracy)

        print(f"Depth: {depth}, Test accuracy: {accuracy:.3f}")

### What did I learn?

| Function | Purpose |
|---|---|
| `mlflow.set_experiment()` | Selects the experiment |
| `mlflow.start_run()` | Starts a tracking run |
| `mlflow.log_param()` | Records a parameter |
| `mlflow.log_metric()` | Records a metric |

Two runs are created: one for depth 1 and another for depth 2.

I can compare their recorded parameters and accuracy in the MLflow experiment interface.

**Note:** If the experiment path causes a permission error, use an experiment associated with your Databricks notebook or a path permitted by your workspace.

### Step 3: Open the MLflow experiment

1. Open your Databricks workspace.
2. Open the Machine Learning notebook you are using.
3. Find the experiment associated with `/Shared/scm-ml-practice`, if the experiment was created successfully.
4. Open the experiment's run list.
5. Compare the `max_depth` parameter and `test_accuracy` metric for both runs.

The exact navigation may vary depending on your Databricks workspace interface.

## Topic 8: Saving and Loading a Machine Learning Model

**Objective:** Learn how to save a trained model and reuse it later without training it again.

In our SCM example, the model predicts whether inventory stock is `low` or `normal`.

| Term | Meaning |
|---|---|
| Train | Teach the model using data |
| Save | Store the trained model |
| Load | Restore the saved model |
| Predict | Use the restored model on new data |

In [0]:
from sklearn.tree import DecisionTreeClassifier
import joblib

model_saved = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

model_saved.fit(X_train, y_train)

joblib.dump(model_saved, "scm_stock_model.pkl")

print("Model saved successfully")

In [0]:
loaded_model = joblib.load("scm_stock_model.pkl")

print("Model loaded successfully")

In [0]:
import pandas as pd

new_inventory = pd.DataFrame({
    "available_quantity": [5, 18, 40],
    "daily_demand": [4, 3, 4],
    "lead_time_days": [5, 3, 3]
})

new_inventory["days_of_stock"] = (
    new_inventory["available_quantity"] / new_inventory["daily_demand"]
)

predictions = loaded_model.predict(new_inventory)

for qty, status in zip(
    new_inventory["available_quantity"],
    predictions
):
    print(f"Quantity: {qty}, Predicted status: {status}")

from sklearn.tree import export_text

tree_rules = export_text(
    loaded_model,
    feature_names=[
        "available_quantity",
        "daily_demand",
        "lead_time_days",
        "days_of_stock"
    ]
)
print("\nTree rules:")
print(tree_rules)

### What did I learn?

| Function | Purpose |
|---|---|
| `joblib.dump()` | Saves a trained model to a file |
| `joblib.load()` | Loads a saved model from a file |
| `predict()` | Predicts labels using the loaded model |

**Important:** A saved model should only be loaded from a trusted source. The model file and its dependencies should be managed carefully.

**Next topic:** Introduction to a complete Machine Learning workflow.

## Topic 9: End-to-End Machine Learning Workflow

**Objective:** Understand the complete process of building and evaluating a Machine Learning model.

| Step | Activity | Purpose |
|---:|---|---|
| 1 | Prepare data | Create and clean the dataset |
| 2 | Select features and target | Separate inputs (`X`) and output (`y`) |
| 3 | Split data | Create training and test sets |
| 4 | Train model | Learn patterns using training data |
| 5 | Predict | Generate predictions for test data |
| 6 | Evaluate | Measure model performance |
| 7 | Track experiments | Record parameters and metrics using MLflow |
| 8 | Save model | Store the trained model for reuse |
| 9 | Load and predict | Reuse the saved model on new data |

We have practiced these steps using our SCM inventory example.

In [0]:
import pandas as pd
import mlflow
import joblib

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Prepare data
data = {
    "available_quantity": [3, 5, 8, 10, 12, 15, 20, 25, 30, 40, 50, 60],
    "stock_status": [
        "low", "low", "low", "low", "low", "low",
        "normal", "normal", "normal", "normal", "normal", "normal"
    ]
}
df = pd.DataFrame(data)

# 2. Select features and target
X = df[["available_quantity"]]
y = df["stock_status"]

# 3. Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# 4. Train model
model_final = DecisionTreeClassifier(max_depth=2, random_state=42)
model_final.fit(X_train, y_train)

# 5. Predict
predictions = model_final.predict(X_test)

# 6. Evaluate
print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions, zero_division=0))

# 7. Save model
joblib.dump(model_final, "scm_stock_model_final.pkl")
print("Model saved.")

### What did I learn?

- Machine Learning involves multiple steps, not just training a model.
- The test set helps evaluate predictions on held-out records.
- MLflow can track experiments and their results.
- A saved model can be loaded later for predictions.

**Important:** This example uses only 12 synthetic records. It demonstrates the workflow, not a production-ready inventory solution.

### Next topic

**Model deployment basics:** how a trained model can be made available to another application or user.

## Topic 10: Machine Learning Model Deployment

**Objective:** Understand how a trained Machine Learning model can be used by another application.

In our SCM example, the model predicts whether inventory stock is `low` or `normal`.

| Step | Activity | Purpose |
|---|---|---|
| 1 | Train the model | Learn patterns from historical data |
| 2 | Save the model | Store it for later use |
| 3 | Load the model | Make the saved model available |
| 4 | Provide input | Send a new inventory quantity |
| 5 | Predict | Return the predicted stock status |

**Deployment** means making a trained model available for use outside the training workflow, such as through an API or an application.

In [0]:
import joblib
import pandas as pd

# Load the previously saved model
loaded_model = joblib.load("scm_stock_model_final.pkl")

# Create a reusable prediction function
def predict_stock_status(quantity):
    input_data = pd.DataFrame({
        "available_quantity": [quantity]
    })

    prediction = loaded_model.predict(input_data)
    return prediction[0]

# Test the function
print(predict_stock_status(5))
print(predict_stock_status(40))

### What did I learn?

| Code | Purpose |
|---|---|
| `joblib.load()` | Loads the trained model |
| `predict_stock_status()` | Wraps prediction logic in a reusable function |
| `predict()` | Predicts the class for a new quantity |
| `return prediction[0]` | Returns the predicted class for one input |

The predicted values depend on the model saved in the previous exercise.

### Real-world deployment options

| Deployment option | Example use |
|---|---|
| REST API | An inventory application sends a quantity and receives a prediction |
| Batch inference | Predict stock status for thousands of inventory records at once |
| Databricks Model Serving | Expose a supported model through a managed serving endpoint |

**Important:** This exercise only simulates prediction logic. It does not create a live API or serving endpoint.

### Next topic

**Batch inference:** use a trained model to predict stock status for many inventory records at once.

## Topic 11: Batch Inference

**Objective:** Use a trained Machine Learning model to predict stock status for multiple inventory records at once.

| Term | Meaning |
|---|---|
| Single prediction | Predict one inventory record |
| Batch inference | Predict many records in one operation |
| Input | Inventory quantities |
| Output | Predicted stock status for each record |

**SCM example:** Predict whether several computer parts have `low` or `normal` stock using the saved model.

In [0]:
import pandas as pd
import joblib

# Load the saved model
loaded_model = joblib.load("scm_stock_model_final.pkl")

# Multiple inventory records
batch_inventory = pd.DataFrame({
    "available_quantity": [3, 10, 18, 25, 40, 60]
})

# Predict all records in one operation
batch_inventory["predicted_status"] = loaded_model.predict(
    batch_inventory[["available_quantity"]]
)

batch_inventory

### What did I learn?

| Step | Explanation |
|---|---|
| Load model | Reuse the trained model |
| Create batch | Prepare multiple inventory quantities |
| Call `predict()` | Predict all records in one operation |
| View results | Display each quantity with its predicted status |

### Learning outcome

Batch inference allows us to classify many inventory records without calling the prediction function separately for each record.

**Important:** These predictions demonstrate the technique using our small synthetic dataset. They are not a production inventory policy.

### Next topic

**Model monitoring basics:** why model performance and prediction quality should be checked after deployment.

## Topic 12: Model Monitoring Basics

**Objective:** Understand why a Machine Learning model should be monitored after deployment.

A model may perform well during training but produce poor predictions later if real-world data changes.

| Concept | Meaning | SCM example |
|---|---|---|
| Model monitoring | Checking model behavior over time | Review stock-status predictions |
| Data drift | Input data distribution changes | Typical inventory quantities increase |
| Prediction monitoring | Reviewing predicted outputs | Count `low` and `normal` predictions |
| Model evaluation | Compare predictions with actual labels when available | Check whether predicted stock status was correct |

**Note:** Data drift can be a warning sign, but it does not automatically mean model performance has declined.

In [0]:
# Count predicted stock statuses
print(
    batch_inventory["predicted_status"].value_counts()
)

# Review the quantities and predictions
batch_inventory

In [0]:
print(
    batch_inventory["available_quantity"].describe()
)

### What did I learn?

- Monitoring helps us observe model inputs and predictions after deployment.
- Data drift means the input data distribution has changed.
- Prediction counts help us understand what the model is producing.
- Actual labels are needed to directly evaluate prediction correctness.

Our six synthetic records are only a practice example; they are not enough to establish real-world drift.

**Next topic:** Improve the inventory dataset and build a more realistic Machine Learning exercise.

## Topic 13: Improving the SCM Dataset

**Objective:** Understand how additional business features can help represent an inventory problem.

| Feature | Meaning |
|---|---|
| `available_quantity` | Current stock quantity |
| `daily_demand` | Expected units consumed per day |
| `lead_time_days` | Days required for supplier delivery |
| `stock_status` | Target class: `low` or `normal` |

We will create a small synthetic dataset for practice. Its labels are illustrative and do not represent a real company's inventory policy.

In [0]:
import pandas as pd

df_scm = pd.DataFrame({
    "available_quantity": [3, 5, 8, 10, 12, 15, 20, 25, 30, 40, 50, 60],
    "daily_demand":       [5, 4, 6, 3, 4, 5, 2, 3, 2, 4, 3, 2],
    "lead_time_days":     [7, 5, 6, 4, 5, 3, 4, 3, 2, 3, 2, 2],
    "stock_status": [
        "low", "low", "low", "low", "low", "low",
        "normal", "normal", "normal", "normal", "normal", "normal"
    ]
})

df_scm

In [0]:
df_scm["days_of_stock"] = (
    df_scm["available_quantity"] / df_scm["daily_demand"]
)

df_scm

### Learning outcome

- A model can use multiple features.
- Feature engineering creates useful inputs from existing data.
- Days of stock relates inventory quantity to demand.
- Additional features do not automatically improve model performance; they must be evaluated.

**Important:** The dataset contains only 12 records. It is suitable for learning Python and ML concepts, not for reliable production predictions.

**Next topic:** Build a complete ML pipeline using preprocessing, model training, and evaluation.

## Topic 14: End-to-End ML Pipeline

An ML pipeline organizes the steps required to train and evaluate a model.

| Step | Purpose |
|---|---|
| 1. Select features and target | Define inputs and output |
| 2. Split data | Separate training and testing records |
| 3. Train model | Learn patterns from training data |
| 4. Predict | Predict stock status for test records |
| 5. Evaluate | Measure model performance |

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Select features and target
X = df_scm[
    ["available_quantity", "daily_demand", "lead_time_days", "days_of_stock"]
]
y = df_scm["stock_status"]

# 2. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# 3. Train the model
model = DecisionTreeClassifier(max_depth=2, random_state=42)
model.fit(X_train, y_train)

# 4. Predict
y_pred = model.predict(X_test)

# 5. Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

### Learning outcome

- The workflow can be rerun when the dataset changes.
- The model learns from training records and is evaluated on test records.
- Accuracy alone is not enough to establish production readiness.

**Important:** This dataset has only 12 synthetic records. Results can change significantly with different splits and are for practice only.

## Topic 15: Scikit-learn Pipeline

A Pipeline connects multiple ML steps into a single workflow.

| Component | Purpose |
|---|---|
| `SimpleImputer` | Fills missing feature values |
| `DecisionTreeClassifier` | Learns patterns and predicts stock status |
| `Pipeline` | Connects the steps and manages their execution |

**Benefit:** Preprocessing is learned from the training data, helping prevent data leakage.

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Select features and target
X = df_scm[
    ["available_quantity", "daily_demand", "lead_time_days", "days_of_stock"]
]
y = df_scm["stock_status"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Create the pipeline
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", DecisionTreeClassifier(max_depth=2, random_state=42))
])

# Train the complete pipeline
pipeline.fit(X_train, y_train)

# Predict and evaluate
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

### Learning outcome

- `fit()` trains the pipeline, including its preprocessing step.
- `predict()` applies the same preprocessing before predicting.
- Missing values can be handled automatically.
- The pipeline can be saved and reused as one object.

**Note:** The small synthetic dataset is for practice only; predictions are not reliable business recommendations.

## Topic 16: Save and Load the ML Pipeline

| Operation | Purpose |
|---|---|
| `joblib.dump()` | Saves the trained pipeline |
| `joblib.load()` | Loads the saved pipeline |
| `predict()` | Predicts using the loaded model |

**Key point:** Saving the complete pipeline preserves both preprocessing and model steps.

In [0]:
import joblib

# Save the trained pipeline
joblib.dump(pipeline, "scm_inventory_pipeline.pkl")

print("Pipeline saved successfully")

In [0]:
# Load the saved pipeline
loaded_pipeline = joblib.load("scm_inventory_pipeline.pkl")

print("Pipeline loaded successfully")

In [0]:
prediction = loaded_pipeline.predict(X_test)

print(prediction)
X_test


### Learning outcome

- The trained pipeline can be saved to a file.
- The saved pipeline can be loaded into a new Python session.
- Predictions can be made without retraining.

**Important:** Only load trusted model files. The `.pkl` file is a serialized Python object, not a portable model format for every platform.

## Topic 17: Batch Inference

Batch inference means generating predictions for multiple records in one execution.

| Type | Example |
|---|---|
| Single prediction | Predict stock status for one item |
| Batch inference | Predict stock status for 1000 inventory items |
| Streaming inference | Predict as new events arrive continuously |

In [0]:
import pandas as pd

batch_stock = pd.DataFrame({
    "available_quantity": [5, 12, 25, 50],
    "daily_demand": [4, 3, 5, 2],
    "lead_time_days": [6, 5, 4, 3],
    "days_of_stock": [1.25, 4.0, 5.0, 25.0]
})

batch_stock

In [0]:
batch_predictions = loaded_pipeline.predict(batch_stock)

batch_stock["predicted_stock_status"] = batch_predictions

batch_stock

### Learning outcome

- One model can predict for many records in a single call.
- Predictions can be added to the original DataFrame.
- The output can be written to a CSV file or stored in a database for downstream processing.

**Important:** These predictions are only for practice because the model was trained on 12 synthetic records.

In [0]:
batch_stock.to_csv("scm_batch_predictions.csv", index=False)

print("Batch predictions saved successfully")

## Topic 18: Inventory Monitoring Report

An inventory monitoring report helps identify items that may require attention.

| Column | Purpose |
|---|---|
| `available_quantity` | Current stock |
| `daily_demand` | Expected daily consumption |
| `days_of_stock` | Estimated days until stock runs out |
| `predicted_stock_status` | ML model prediction |
| `inventory_risk` | Business rule based on days of stock |

In [0]:
# Create a copy for reporting
inventory_report = batch_stock.copy()

# Apply a simple business rule
inventory_report["inventory_risk"] = inventory_report[
    "days_of_stock"
].apply(
    lambda days: "High" if days <= 2
    else "Medium" if days <= 5
    else "Low"
)

inventory_report

In [0]:
risk_summary = inventory_report["inventory_risk"].value_counts()

print(risk_summary)

In [0]:
inventory_report.to_csv(
    "scm_inventory_monitoring_report.csv",
    index=False
)

print("Inventory monitoring report saved")

### Learning outcome

- ML predictions and business rules can be used together.
- Business rules classify inventory risk based on days of stock.
- Reports help business teams identify items requiring attention.

**Important:** The risk thresholds are illustrative, not universal inventory policies. Actual thresholds should consider demand variability, supplier lead time, and safety stock.

The inventory risk is calculated using a business rule, not predicted by the ML model.

## Topic 19: Business Rules vs ML Predictions

| Aspect | Business Rule | ML Prediction |
|---|---|---|
| Logic | Explicitly defined by a person | Learns patterns from training data |
| Example | High risk if days of stock <= 2 | Predicts low or normal stock status |
| Changes | Update the rule manually | Retrain with suitable new data |
| Output | Determined by the rule | Determined by the trained model |
| Best use | Clear, known policies | Patterns that are difficult to define manually |

In [0]:
comparison = batch_stock.copy()

# ML prediction
comparison["ml_prediction"] = loaded_pipeline.predict(
    comparison[
        ["available_quantity", "daily_demand",
         "lead_time_days", "days_of_stock"]
    ]
)

# Business rule
comparison["business_rule"] = comparison["days_of_stock"].apply(
    lambda days: "low" if days <= 2 else "normal"
)

comparison

### How to interpret the results

- `ml_prediction` comes from the trained Decision Tree.
- `business_rule` comes from a manually defined threshold.
- The two outputs may agree or disagree.
- A disagreement should trigger investigation, not an automatic assumption that one is correct.

**Important:** The threshold of 2 days is illustrative. The ML model and synthetic training data are also for learning purposes, not production inventory decisions.

In [0]:
comparison["outputs_differ"] = (
    comparison["ml_prediction"] != comparison["business_rule"]
)

comparison[
    ["days_of_stock", "ml_prediction",
     "business_rule", "outputs_differ"]
]

## Topic 20: Baseline Model vs ML Model

A baseline is a simple reference used to judge whether a more complex model adds value.

| Model | Approach |
|---|---|
| Baseline | Always predicts the most frequent training label |
| Decision Tree | Learns patterns from inventory features |

We compare both models on the same test records.

In [0]:
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Baseline model
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

# Decision Tree model
tree = DecisionTreeClassifier(max_depth=2, random_state=42)
tree.fit(X_train, y_train)

tree_predictions = tree.predict(X_test)

# Compare accuracy
print("Baseline accuracy:",
      accuracy_score(y_test, baseline_predictions))

print("Decision Tree accuracy:",
      accuracy_score(y_test, tree_predictions))

### Interpret the results

- Baseline accuracy shows how well a simple majority-class strategy performs.
- Decision Tree accuracy shows how well the trained tree predicts the test records.
- If the tree performs better, it may be learning useful patterns.
- If both scores are equal, the tree has not demonstrated an accuracy improvement on this test split.

**Important:** Your dataset has only 12 synthetic records. These results are for learning and do not establish real-world model quality.

## Topic 21: Precision, Recall, and F1-Score

| Metric | Meaning in inventory classification |
|---|---|
| Precision | Of the items predicted as `low`, how many are actually `low`? |
| Recall | Of the actual `low` items, how many did the model identify? |
| F1-score | Balances precision and recall |
| Accuracy | Percentage of all predictions that are correct |

**Business perspective:** Missing an item that is genuinely low in stock may be costly. Recall for the `low` class can therefore be especially important.

In [0]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = tree.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=["low", "normal"]))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=["low", "normal"],
    zero_division=0
))

### How to interpret the results

- Check the `low` row in the classification report.
- Low recall means some genuinely low-stock records were missed.
- Low precision means some records predicted as low were actually normal.
- F1-score helps balance precision and recall.

**Important:** With only 12 synthetic records, the test set is very small. These metrics can change significantly with different data splits.

In [0]:
from sklearn.metrics import recall_score

low_stock_recall = recall_score(
    y_test,
    y_pred,
    pos_label="low",
    zero_division=0
)

print("Low-stock recall:", round(low_stock_recall, 2))

## Topic 22: False Positives and False Negatives

Assume `low` is the positive class.

| Outcome | Meaning | Inventory example |
|---|---|---|
| True Positive (TP) | Actual low, predicted low | Correctly identifies low stock |
| True Negative (TN) | Actual normal, predicted normal | Correctly identifies normal stock |
| False Positive (FP) | Actual normal, predicted low | Unnecessary low-stock alert |
| False Negative (FN) | Actual low, predicted normal | Misses a low-stock item |

**Business impact:** A false negative can be costly if it causes a stockout. A false positive may lead to unnecessary checks or replenishment.

In [0]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["low", "normal"]
)

print(cm)

### How to read the output

The matrix uses this order:

- Rows: actual labels (`low`, `normal`)
- Columns: predicted labels (`low`, `normal`)

| | Predicted low | Predicted normal |
|---|---:|---:|
| Actual low | TP | FN |
| Actual normal | FP | TN |

The diagonal contains correct predictions. The off-diagonal cells contain errors.

In [0]:
tp = cm[0, 0]
fn = cm[0, 1]
fp = cm[1, 0]
tn = cm[1, 1]

print("True Positives:", tp)
print("False Negatives:", fn)
print("False Positives:", fp)
print("True Negatives:", tn)

## Topic 23: Hyperparameter Tuning

Hyperparameters are settings chosen before training a model.

For a Decision Tree:
- `max_depth=1`: simpler tree
- `max_depth=2`: more complex tree
- `max_depth=3`: potentially more complex tree

The goal is to find a suitable setting without overfitting.

In [0]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

for depth in [1, 2, 3]:
    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    print(
        f"Depth: {depth}, "
        f"Test Accuracy: {accuracy_score(y_test, predictions):.2f}"
    )

## Topic 24: Machine Learning — Final Recap

| Step | What we learned |
|---|---|
| 1. Data preparation | Create a dataset using pandas |
| 2. Features and target | Separate inputs (`X`) and output (`y`) |
| 3. Train/test split | Separate training and testing data |
| 4. Model training | Train a Decision Tree |
| 5. Evaluation | Check accuracy and classification metrics |
| 6. Cross-validation | Evaluate across multiple data splits |
| 7. Pipeline | Combine preprocessing and model training |
| 8. Model persistence | Save and load a trained model |
| 9. Batch inference | Predict for multiple inventory records |
| 10. Monitoring | Review inventory predictions and risk |

### Final takeaway

Machine learning uses historical data to train a model, evaluates its performance, and uses the trained model to make predictions on new data.

Our SCM dataset is synthetic and small, so it is suitable for learning, not production decisions.